In [26]:
from google.colab import drive
import pandas as pd
import re
drive.mount('/content/drive')
file_path = '/content/drive/My Drive/NLP/genap.csv'
df = pd.read_csv(file_path, delimiter=';')
print("Original Data Head:")
print(df.head(10))

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
Original Data Head:
    No                                               Data
0  1.0  KOMPAS.com – Banjir dan longsor yang melanda A...
1  NaN                                                NaN
2  NaN  Sumber: https://lestari.kompas.com/read/2026/0...
3  NaN                                                NaN
4  NaN                                                NaN
5  NaN                   Membership: https://kmp.im/plus6
6  NaN             Download aplikasi: https://kmp.im/app6


In [27]:
def clean_text(text):
    if pd.isna(text):
        return ""
    text = str(text) # Convert to string but no longer lowercase here
    text = re.sub(r'http\S+|www\S+|https\S+', '', text, flags=re.MULTILINE) # Remove URLs
    text = re.sub(r'membership:\S+|download aplikasi:\S+', '', text, flags=re.MULTILINE) # Remove membership/download app lines
    text = re.sub(r'[^a-zA-Z\s]', '', text) # Remove non-alphabetic characters except spaces
    text = re.sub(r'\s+', ' ', text).strip() # Remove extra spaces
    return text

df['Cleaned_Data'] = df['Data'].apply(clean_text)
print("\nCleaned Data Head:")
print(df[['Data', 'Cleaned_Data']].head(10))


Cleaned Data Head:
                                                Data  \
0  KOMPAS.com – Banjir dan longsor yang melanda A...   
1                                                NaN   
2  Sumber: https://lestari.kompas.com/read/2026/0...   
3                                                NaN   
4                                                NaN   
5                   Membership: https://kmp.im/plus6   
6             Download aplikasi: https://kmp.im/app6   

                                        Cleaned_Data  
0  KOMPAScom Banjir dan longsor yang melanda Aceh...  
1                                                     
2                                             Sumber  
3                                                     
4                                                     
5                                         Membership  
6                                  Download aplikasi  


In [28]:
import nltk
nltk.download('punkt')
nltk.download('punkt_tab') # Download the missing resource

def tokenize_text(text):
    return nltk.word_tokenize(text)

df['Tokens'] = df['Cleaned_Data'].apply(tokenize_text)
print("\nData with Tokens:")
print(df[['Cleaned_Data', 'Tokens']].head(10))


Data with Tokens:
                                        Cleaned_Data  \
0  KOMPAScom Banjir dan longsor yang melanda Aceh...   
1                                                      
2                                             Sumber   
3                                                      
4                                                      
5                                         Membership   
6                                  Download aplikasi   

                                              Tokens  
0  [KOMPAScom, Banjir, dan, longsor, yang, meland...  
1                                                 []  
2                                           [Sumber]  
3                                                 []  
4                                                 []  
5                                       [Membership]  
6                               [Download, aplikasi]  


[nltk_data] Downloading package punkt to /root/nltk_data...
[nltk_data]   Package punkt is already up-to-date!
[nltk_data] Downloading package punkt_tab to /root/nltk_data...
[nltk_data]   Package punkt_tab is already up-to-date!


In [29]:
normalization_dict = {
    "gk": "tidak",
    "ga": "tidak",
    "yg": "yang",
    "dgn": "dengan",
    "aja": "saja"
}

def normalize(tokens):
    return [normalization_dict.get(word, word) for word in tokens]

df['normalized'] = df['Tokens'].apply(normalize)

print("\nData after Normalization:")
print(df[['Cleaned_Data', 'Tokens', 'normalized']].head(10))


Data after Normalization:
                                        Cleaned_Data  \
0  KOMPAScom Banjir dan longsor yang melanda Aceh...   
1                                                      
2                                             Sumber   
3                                                      
4                                                      
5                                         Membership   
6                                  Download aplikasi   

                                              Tokens  \
0  [KOMPAScom, Banjir, dan, longsor, yang, meland...   
1                                                 []   
2                                           [Sumber]   
3                                                 []   
4                                                 []   
5                                       [Membership]   
6                               [Download, aplikasi]   

                                          normalized  
0  [KOMPAScom, Banji

In [30]:
import nltk
nltk.download('stopwords')
from nltk.corpus import stopwords

stop_words_id = set(stopwords.words('indonesian'))

def remove_stopwords(tokens):
    return [word for word in tokens if word not in stop_words_id]

df['Tokens_no_stopwords'] = df['normalized'].apply(remove_stopwords)

print("\nData after Stopword Removal:")
print(df[['normalized', 'Tokens_no_stopwords']].head(10))


Data after Stopword Removal:
                                          normalized  \
0  [KOMPAScom, Banjir, dan, longsor, yang, meland...   
1                                                 []   
2                                           [Sumber]   
3                                                 []   
4                                                 []   
5                                       [Membership]   
6                               [Download, aplikasi]   

                                 Tokens_no_stopwords  
0  [KOMPAScom, Banjir, longsor, melanda, Aceh, me...  
1                                                 []  
2                                           [Sumber]  
3                                                 []  
4                                                 []  
5                                       [Membership]  
6                               [Download, aplikasi]  


[nltk_data] Downloading package stopwords to /root/nltk_data...
[nltk_data]   Package stopwords is already up-to-date!


In [31]:
df['case_folding'] = df['Cleaned_Data'].str.lower()
print("\nData after Case Folding (explicitly shown):")
print(df[['Cleaned_Data', 'case_folding']].head(10))


Data after Case Folding (explicitly shown):
                                        Cleaned_Data  \
0  KOMPAScom Banjir dan longsor yang melanda Aceh...   
1                                                      
2                                             Sumber   
3                                                      
4                                                      
5                                         Membership   
6                                  Download aplikasi   

                                        case_folding  
0  kompascom banjir dan longsor yang melanda aceh...  
1                                                     
2                                             sumber  
3                                                     
4                                                     
5                                         membership  
6                                  download aplikasi  


In [32]:
pip install Sastrawi

In [33]:
from Sastrawi.Stemmer.StemmerFactory import StemmerFactory

# Create stemmer
factory = StemmerFactory()
stemmer = factory.create_stemmer()

def stem_text(tokens):
    stemmed_tokens = [stemmer.stem(word) for word in tokens]
    return stemmed_tokens

df['Stemmed_Tokens'] = df['Tokens_no_stopwords'].apply(stem_text)

print("\nData after Stemming:")
print(df[['Tokens_no_stopwords', 'Stemmed_Tokens']].head(10))


Data after Stemming:
                                 Tokens_no_stopwords  \
0  [KOMPAScom, Banjir, longsor, melanda, Aceh, me...   
1                                                 []   
2                                           [Sumber]   
3                                                 []   
4                                                 []   
5                                       [Membership]   
6                               [Download, aplikasi]   

                                      Stemmed_Tokens  
0  [kompascom, banjir, longsor, landa, aceh, ting...  
1                                                 []  
2                                           [sumber]  
3                                                 []  
4                                                 []  
5                                       [membership]  
6                               [download, aplikasi]  


In [34]:
from Sastrawi.Stemmer.StemmerFactory import StemmerFactory

# Create lemmatizer (Sastrawi's StemmerFactory can also create a Lemmatizer based on the same logic for Indonesian)
factory = StemmerFactory()
lemmatizer = factory.create_stemmer() # Sastrawi's 'stemmer' acts as a lemmatizer for Indonesian

def lemmatize_text(tokens):
    lemmatized_tokens = [lemmatizer.stem(word) for word in tokens]
    return lemmatized_tokens

df['Lemmatized_Tokens'] = df['Tokens_no_stopwords'].apply(lemmatize_text)

print("\nData after Lemmatization:")
print(df[['Tokens_no_stopwords', 'Stemmed_Tokens', 'Lemmatized_Tokens']].head(10))


Data after Lemmatization:
                                 Tokens_no_stopwords  \
0  [KOMPAScom, Banjir, longsor, melanda, Aceh, me...   
1                                                 []   
2                                           [Sumber]   
3                                                 []   
4                                                 []   
5                                       [Membership]   
6                               [Download, aplikasi]   

                                      Stemmed_Tokens  \
0  [kompascom, banjir, longsor, landa, aceh, ting...   
1                                                 []   
2                                           [sumber]   
3                                                 []   
4                                                 []   
5                                       [membership]   
6                               [download, aplikasi]   

                                   Lemmatized_Tokens  
0  [kompascom, banji

In [35]:
print("\nAll Processed Data (Head):")
print(df[['Data', 'Cleaned_Data', 'Tokens', 'normalized', 'Tokens_no_stopwords', 'Stemmed_Tokens', 'Lemmatized_Tokens']].head(10))


All Processed Data (Head):
                                                Data  \
0  KOMPAS.com – Banjir dan longsor yang melanda A...   
1                                                NaN   
2  Sumber: https://lestari.kompas.com/read/2026/0...   
3                                                NaN   
4                                                NaN   
5                   Membership: https://kmp.im/plus6   
6             Download aplikasi: https://kmp.im/app6   

                                        Cleaned_Data  \
0  KOMPAScom Banjir dan longsor yang melanda Aceh...   
1                                                      
2                                             Sumber   
3                                                      
4                                                      
5                                         Membership   
6                                  Download aplikasi   

                                              Tokens  \
0  [KOMPAScom, Ban